# 微台秒級資料品質與價格預測核對
本機伴隨證據：29份TAIFEX公開ZIP、30780根1m對帳及原生模型重新預測。
需要保留 `.fubon-dev/tick-research/taifex-tmf-v1` 與 `seconds-v1-verified` 產物，並使用server learning環境。
下列cell以專案CPython實際執行並保存stdout；不是瀏覽器或Paper策略執行證據。
原始市場資料不隨notebook提交，沒有歷史BBO、沒有個人心理標籤，不做獲利保證。


In [1]:
from pathlib import Path
import gzip, hashlib, json, sys
import numpy as np
import lightgbm as lgb

repo=Path.cwd().resolve()
while not (repo/'server/fubon_bridge/microstructure_data.py').exists():
    if repo.parent==repo:
        raise RuntimeError('請在保留本輪本機資料的repository內執行')
    repo=repo.parent
sys.path[:0]=[str(repo/'server'),str(repo/'scripts')]
from fubon_bridge.research import digest
from fubon_bridge.candle_patterns import boundary
from importlib import import_module
metrics=import_module('analyze-tmf-seconds').metrics
art=repo/'.fubon-dev/reports/seconds-v1-verified'
src=repo/'.fubon-dev/tick-research/taifex-tmf-v1'
manifest=json.loads((src/'manifest.json').read_text())
audit=json.loads((art/'audit.json').read_text())
frozen=json.loads((art/'frozen-spec.json').read_text())
assert digest({k:v for k,v in frozen.items() if k!='id'})==frozen['id']
assert hashlib.sha256((src/'manifest.json').read_bytes()).hexdigest()==audit['manifestSha256']
for r in manifest['archives']:
    assert hashlib.sha256((src/r['archive']).read_bytes()).hexdigest()==r['quality']['archiveSha256']
for r in audit['verifiedSeconds']:
    assert hashlib.sha256((src/r['file']).read_bytes()).hexdigest()==r['sha256']
assert audit['quoteRows']==0 and not audit['executionAllowed'] and not audit['adoptionAllowed']
print(json.dumps(dict(frozenId=frozen['id'],archives=len(manifest['archives']),
    reportingDates=audit['tradingDates'],totals=audit['totals'],quoteRows=0),ensure_ascii=False,indent=2))


{
  "frozenId": "1c9ee21ba75f270cf5dfff5e2b7e484ea6bd3f09c07ee573ff328a7d2e114bce",
  "archives": 29,
  "reportingDates": [
    "2026-10-05",
    "2026-10-06",
    "2026-10-07",
    "2026-10-08",
    "2026-10-12",
    "2026-10-13",
    "2026-10-14",
    "2026-10-15",
    "2026-10-16",
    "2026-10-19",
    "2026-10-20",
    "2026-10-21",
    "2026-10-22",
    "2026-10-27",
    "2026-10-28",
    "2026-10-29",
    "2026-10-30",
    "2026-11-02",
    "2026-11-03",
    "2026-11-04",
    "2026-11-05",
    "2026-11-06",
    "2026-11-09",
    "2026-11-10",
    "2026-11-11",
    "2026-11-12",
    "2026-11-13",
    "2026-11-16",
    "2026-11-17"
  ],
  "totals": {
    "prints": 11887069,
    "quantityBothSides": 31149940,
    "oddQuantityRows": 0,
    "auctionPrints": 210,
    "tradedContractSeconds": 2034668,
    "multiplePrintSeconds": 1387055,
    "multiplePriceSeconds": 1240772,
    "excludedNonMonthlyRows": 3634
  },
  "quoteRows": 0
}


In [2]:
for session,record in audit['reconciliation'].items():
    checks=record['checks'];n=checks['matchedMinutes']
    assert all(checks[k]==n for k in ('highLow','volumeHalf','openWithinFirstSecond','closeWithinLastSecond'))
    assert checks['volumeRaw']==0
    print(session,json.dumps(dict(matchedMinutes=n,rates=record['matchRates']),ensure_ascii=False))
print('首末秒涵蓋只為必要條件，不證明同秒先後；B+S/2只對本批資料完成對帳。')


regular {"matchedMinutes": 8100, "rates": {"highLow": 1.0, "volumeRaw": 0.0, "volumeHalf": 1.0, "openWithinFirstSecond": 1.0, "closeWithinLastSecond": 1.0}}
afterhours {"matchedMinutes": 22680, "rates": {"highLow": 1.0, "volumeRaw": 0.0, "volumeHalf": 1.0, "openWithinFirstSecond": 1.0, "closeWithinLastSecond": 1.0}}
首末秒涵蓋只為必要條件，不證明同秒先後；B+S/2只對本批資料完成對帳。


In [3]:
results={}
for session in ('regular','afterhours'):
    result=json.loads((art/f'{session}-result.json').read_text())
    with gzip.open(art/f'{session}-observations.jsonl.gz','rt') as stream:
        rows=[json.loads(line) for line in stream]
    cutoff=boundary(result['dateSplit']['validation'][0])
    fit=[r for r in rows if r['day'] in result['dateSplit']['fit'] and r['labelEnd']<cutoff]
    assert len(rows)==result['validObservations']
    results[session]=dict(validObservations=len(rows),fitObservations=len(fit),blocked=result.get('blocked'))
    if not result['models']:
        assert len(fit)<1000 and result.get('blocked')
        continue
    for variant,record in result['models'].items():
        model_path=art/record['file']
        assert hashlib.sha256(model_path.read_bytes()).hexdigest()==record['sha256']
        booster=lgb.Booster(model_file=str(model_path))
        x=np.array([r['coarse']+(r['micro'] if variant=='seconds' else []) for r in rows])
        prediction=booster.predict(x,num_threads=1)
        assert record['maximumLabelEnd']==max(r['labelEnd'] for r in fit)<cutoff
        for phase,days in result['dateSplit'].items():
            mask=np.array([r['day'] in days and (phase!='fit' or r['labelEnd']<cutoff) for r in rows])
            subset=[r for r,keep in zip(rows,mask,strict=True) if keep]
            actual=metrics(subset,prediction[mask])
            expected=result['metrics'][phase][variant]
            assert actual==expected
        results[session][variant]=result['metrics']['reused_history'][variant]
    results[session]['pairedDayBootstrap']=result['pairedDayBootstrap']
    # Independently reproduce the paired-day bootstrap from recorded daily errors.
    c=result['metrics']['reused_history']['coarse']['daily']
    s=result['metrics']['reused_history']['seconds']['daily']
    differences=np.array([b['maeAtr']-a['maeAtr'] for a,b in zip(c,s,strict=True)])
    boot=np.random.default_rng(42).choice(differences,(2000,len(differences)),replace=True).mean(axis=1)
    assert list(np.quantile(boot,[.025,.975]))==result['pairedDayBootstrap']['percentile95']
    assert result['netProfit'] is None and not result['executionAllowed'] and not result['adoptionAllowed']
print(json.dumps(results,ensure_ascii=False,indent=2))
print('價格預測比較未證實可靠改善；缺BBO，不是可成交或成本後淨利回測。')


{
  "regular": {
    "validObservations": 1758,
    "fitObservations": 994,
    "blocked": "探索fit不足10日／1000觀測，或後段日期不足"
  },
  "afterhours": {
    "validObservations": 4259,
    "fitObservations": 2111,
    "blocked": null,
    "coarse": {
      "observations": 1162,
      "days": 6,
      "maeAtr": 0.4131236068540203,
      "dailyMeanMaeAtr": 0.4134445459005103,
      "clippedTargetMaePoints": 19.012236706995846,
      "rawPriceMaePoints": 19.225945401192895,
      "nonzeroDirectionObservations": 1154,
      "directionAccuracy": 0.5043327556325823,
      "daily": [
        {
          "day": "2026-11-06",
          "maeAtr": 0.41033878383389016
        },
        {
          "day": "2026-11-09",
          "maeAtr": 0.40833186780827285
        },
        {
          "day": "2026-11-10",
          "maeAtr": 0.40161829748685773
        },
        {
          "day": "2026-11-11",
          "maeAtr": 0.4228229880254734
        },
        {
          "day": "2026-11-12",
          "maeAtr": 